In [0]:
from pathlib import PurePosixPath
import sys

# Get the current notebook path dynamically (Databricks workspace path)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = PurePosixPath("/Workspace") / PurePosixPath(notebook_path).parents[2]
sys.path.append(str(repo_root))

import json
import datetime
import utils
session = utils.create_session()


In [0]:
# Read all saved location files (wildcard) to avoid relying on a single filename
df = spark.read.json('/Volumes/openaq/bronze_dev/locations/*')
result_list = df.select("sensors", "id").toPandas().to_dict(orient='records')

In [0]:
import datetime
from pathlib import Path

# fetch hourly measurements for the last 24 hours by default
dt_to = datetime.datetime.utcnow()
dt_from = dt_to - datetime.timedelta(days=1)
datetime_to = dt_to.replace(microsecond=0).isoformat() + 'Z'
datetime_from = dt_from.replace(microsecond=0).isoformat() + 'Z'

for item in result_list:
    location_id = item.get('id')
    sensors = item.get('sensors', [])
    for sensor in sensors:
        sensor_id = sensor.get('id')
        if not sensor_id:
            continue
        measurements = utils.get_sensor_measurements(session, sensor_id, datetime_from=datetime_from, datetime_to=datetime_to, limit=100)
        timestamp = datetime.datetime.utcnow().strftime("%Y-%m-%d_%H-%M-%S")
        dir_path = Path(f'/Volumes/openaq/bronze_dev/sensor_data/{location_id}_{sensor_id}')
        dir_path.mkdir(parents=True, exist_ok=True)
        with open(dir_path / f'{sensor_id}_{timestamp}.json', 'w') as f:
            f.write(json.dumps(measurements))

In [0]:
%sql
select * from read_files('/Volumes/openaq/bronze_dev/sensor_data/*')

In [0]:
#need to make sure I am properly working with pagination. so if limit is les than found go fetch all the pages
#save the data in folders of location_id + sensor_id, than inside of the folders set the timestamp as the name of the file or sensor_data+timestamp, it is now quite messy